# Desafio A - Compras públicas e diferenças regionais 
---

## Problema
- A distribuiçao das contrataçoes públicas apresenta difereças a entre as Unidade da Federação?

## Estrategia de resolução 

- Vamos criar um mapa de calor com um filtro de anos, de 2020 a 2026. Será possível visualizar as contratações de cada ano e também o total acumulado do período.

O mapa vai mostrar todos os países e a quantidade de contratações realizadas. Quanto mais forte a cor, maior o número de contratações; quanto mais fraca, menor a quantidade. Assim, fica fácil identificar quais países mais contratam ao longo dos anos.

## Perguntas 
- Quantidade de contratação aentre os estados
- Estados Com mais contratações 
- Estados com menos contratações 
- Estados com maior concetração em diferentes modalidades 
- Os vaalores gastos por cada estaçao por ano analisado + somativo geral nacional 

## Análises 
- Frequencias 
- Mediana 
- Média 
- Variação 
- Maior 
- Minimo 

## Filtros utilizados da API
- Consultar Contratacoes 
- Consultar Itens
- Consualtar resultados itens contratacoes

## Bibliotecas

Utilizaremos principalmente:

- `requests` para realizar as requisições HTTP;
- `pandas` para organizar e analisar os dados;
- `matplotlib` para visualizações simples.

In [ ]:
#%pip install requests pandas matplotlib -q

Note: you may need to restart the kernel to use updated packages.


In [19]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import time

## URLs ultilizadas:

A API do Compras.gov.br possui diferentes módulos.

Alguns exemplos:

```text
Catálogo de materiais
Catálogo de serviços
Pesquisa de preços
Contratações
Atas de Registro de Preços
Contratos
Fornecedores
```

Nesse projeto ultilizaremos o modulo de contratações, fazendo uso dos seguintes endpoints:

```text
GET /modulo-contratacoes/1_consultarContratacoes_PNCP_14133
GET //modulo-contratacoes/2_consultarItensContratacoes_PNCP_14133
GET //modulo-contratacoes/3_consultarResultadoItensContratacoes_PNCP_14133
```

Esse serviço consulta contratações realizadas no contexto da Lei nº 14.133/2021. Esta Lei estabelece normas gerais de licitação e contratação para as Administrações Públicas diretas, autárquicas e fundacionais da União, dos Estados, do Distrito Federal e dos Municípios

A URL base será:

In [3]:
base_url = "https://dadosabertos.compras.gov.br"

endpoint_contratacoes = "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"
endpoint_itens = "/modulo-contratacoes/2_consultarItensContratacoes_PNCP_14133"
endpoint_resultados_itens = "/modulo-contratacoes/3_consultarResultadoItensContratacoes_PNCP_14133"


## Parâmetros da consulta

O endpoint permite utilizar parâmetros para restringir a consulta.

Nesta atividade utilizaremos:

- `anos`: anos dos registros;
- `tamanhoPagina`: número de registros por página;
- `maximoPaginas`: maxímo de registros por página;
- `codigoModalidade`: código da modalidade;



In [62]:
anos = list(range(2020, 2027))          
modalidade = 6                         
tamanho_paginas = 50                 
max_paginas_ano = 20              

parametros = {
    "ano": anos,
    "tamanhoPagina": tamanho_paginas,
    "maximoPaginas": max_paginas_ano,
    "codigoModalidade": modalidade,
}

parametros

{'ano': [2020, 2021, 2022, 2023, 2024, 2025, 2026],
 'tamanhoPagina': 50,
 'maximoPaginas': 20,
 'codigoModalidade': 6}

## Consulta simples dos dados

Antes da coleta completa (2020–2026), fizemos uma consulta pequena para verificar:

- o endpoint estava disponível?
- os campos necessários (UF, valor, modalidade) estão presentes?
- o `codigoModalidade` escolhido retorna registros?

In [16]:
def extrair_informacoes(dados):
    return dados["resultado"]

params_dados = {
    "pagina": 1,
    "tamanhoPagina": 10,
    "dataPublicacaoPncpInicial": "2025-05-01",
    "dataPublicacaoPncpFinal": "2025-05-15",
    "codigoModalidade": modalidade,
}

resposta = requests.get(base_url + endpoint_contratacoes, params=params_dados)

print("Status HTTP:", resposta.status_code)
print("URL consultada:", resposta.url)

registros = extrair_informacoes(resposta.json())
df_simples = pd.json_normalize(registros)

print("Registros encontrados:", len(df_simples))
df_simples.head()

Status HTTP: 200
URL consultada: https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133?pagina=1&tamanhoPagina=10&dataPublicacaoPncpInicial=2025-05-01&dataPublicacaoPncpFinal=2025-05-15&codigoModalidade=6
Registros encontrados: 10


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,78251006900072025,00394502000144-1-003952/2025,2025,3952,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,34103.12,NaN,2025-05-01T08:48:39,2025-05-01T08:48:39,2025-05-01T08:48:39,2025-05-01T08:48:36,2025-05-08T07:59:59,False
1,92826706900152025,14846532000159-1-000021/2025,2025,21,14846532000159,None,67124,CONSELHO DE ARQUITETURA E URBANISMO DO AMAPA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,4100.00,1500.0,2025-05-01T11:08:48,2025-05-01T11:08:48,2025-05-01T11:08:48,2025-05-01T11:08:47,2025-05-08T07:59:59,False
2,78882006900212025,00394502000144-1-003953/2025,2025,3953,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,7451.86,NaN,2025-05-01T11:24:21,2025-05-01T11:24:21,2025-05-01T11:24:21,2025-05-01T11:24:19,2025-05-08T07:59:59,False
3,78790006900502025,00394502000144-1-003954/2025,2025,3954,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,4667.26,4275.7,2025-05-01T12:12:43,2025-05-01T12:12:43,2025-05-01T12:12:43,2025-05-01T12:12:42,2025-05-08T07:59:59,False
4,16000606900072025,00394452000103-1-009791/2025,2025,9791,00394452000103,None,44611,COMANDO DO EXERCITO,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,76274.86,63987.0,2025-05-01T12:17:43,2025-05-01T12:17:43,2025-05-01T12:17:43,2025-05-01T12:17:42,2025-05-13T08:59:59,False


In [35]:
colunas = [
    "idCompra",
    "numeroCompra",
    "anoCompraPncp",
    "modalidadeNome",
    "modalidadeIdPncp",
    "unidadeOrgaoNomeUnidade",
    "orgaoEntidadeRazaoSocial",
    "objetoCompra",
    "valorTotalEstimado",
    "valorTotalHomologado",
    "dataPublicacaoPncp",
    "srp",
]

colunas_existentes = []

for coluna in colunas:
    if coluna in df_simples.columns:
        colunas_existentes.append(coluna)

print("Colunas retornadas pela API:")
print(df_simples.columns.tolist())

print("\nColunas de interesse encontradas:")
print(colunas_existentes)

print("\nColuna da UF:", "unidadeOrgaoUfSigla")

Colunas retornadas pela API:
['idCompra', 'numeroControlePNCP', 'anoCompraPncp', 'sequencialCompraPncp', 'orgaoEntidadeCnpj', 'orgaoSubrogadoCnpj', 'codigoOrgao', 'orgaoEntidadeRazaoSocial', 'orgaoSubrogadoRazaoSocial', 'orgaoEntidadeEsferaId', 'orgaoSubrogadoEsferaId', 'orgaoEntidadePoderId', 'orgaoSubrogadoPoderId', 'unidadeOrgaoCodigoUnidade', 'unidadeSubrogadaCodigoUnidade', 'unidadeOrgaoNomeUnidade', 'unidadeSubrogadaNomeUnidade', 'unidadeOrgaoUfSigla', 'unidadeSubrogadaUfSigla', 'unidadeOrgaoMunicipioNome', 'unidadeSubrogadaMunicipioNome', 'unidadeOrgaoCodigoIbge', 'unidadeSubrogadaCodigoIbge', 'numeroCompra', 'modalidadeIdPncp', 'codigoModalidade', 'modalidadeNome', 'srp', 'modoDisputaIdPncp', 'codigoModoDisputa', 'amparoLegalCodigoPncp', 'amparoLegalNome', 'amparoLegalDescricao', 'informacaoComplementar', 'processo', 'objetoCompra', 'existeResultado', 'orcamentoSigilosoCodigo', 'orcamentoSigilosoDescricao', 'situacaoCompraIdPncp', 'situacaoCompraNomePncp', 'tipoInstrumentoConvo

## 9. Coleta dos dados

* Essa função irá coletar, dados de um determinado ano, de acordo com o `max`.

* seguindo as boas práticas da aula: paginação controlada, `timeout`, interrupção quando a página vier vazia.

In [36]:
def coletar_ano(ano):
    data_inicial = f"{ano}-01-01"
    data_final = f"{ano}-12-31"

    todos_registros = []

    for pagina in range(1, max_paginas_ano + 1):

        params = {
            "pagina": pagina,
            "tamanhoPagina": tamanho_paginas,
            "dataPublicacaoPncpInicial": data_inicial,
            "dataPublicacaoPncpFinal": data_final,
            "codigoModalidade": modalidade,
        }

        resposta = requests.get(
            base_url + endpoint_contratacoes,
            params=params,
            timeout=60
        )

        print(f"Ano {ano} - Página {pagina} - Status: {resposta.status_code}")

        if resposta.status_code != 200:
            break

        registros = extrair_informacoes(resposta.json())

        print(f"Registros encontrados: {len(registros)}")

        if len(registros) == 0:
            break

        todos_registros.extend(registros)

        if len(registros) < tamanho_paginas:
            break

        time.sleep(0.3)

    df_ano = pd.json_normalize(todos_registros)

    if not df_ano.empty:
        df_ano["anoConsulta"] = ano

    return df_ano

In [37]:
dataframes_por_ano = []

for ano in anos:
    print(f"Coletando ano {ano}...")

    df_ano = coletar_ano(ano)

    print(f"Registros coletados em {ano}: {len(df_ano)}")

    dataframes_por_ano.append(df_ano)

df_infos = pd.concat(dataframes_por_ano, ignore_index=True)

print("Total de registros:", len(df_infos))

df_infos.head()

Coletando ano 2020...
Ano 2020 - Página 1 - Status: 200
Registros encontrados: 0
Registros coletados em 2020: 0
Coletando ano 2021...
Ano 2021 - Página 1 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 2 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 3 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 4 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 5 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 6 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 7 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 8 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 9 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 10 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 11 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 12 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 13 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 14 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 15 

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,anoConsulta
0,78333006000012021,00394502000144-1-000001/2021,2021,1,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Não se aplica,49506.60,NaN,2021-08-10T07:54:24,2021-08-10T07:54:24,2021-08-10T07:54:24,2021-08-10T08:00:00,2021-08-13T14:00:00,False,2021
1,15327906000042021,17217985000104-1-000001/2021,2021,1,17217985000104,None,16209,UNIVERSIDADE FEDERAL DE MINAS GERAIS,None,F,...,Não se aplica,23147.09,10684.52,2021-08-10T07:54:38,2021-08-10T07:54:38,2021-08-10T07:54:38,2021-08-10T08:00:00,2021-08-13T15:00:00,False,2021
2,15329206000052021,17217985000104-1-000002/2021,2021,2,17217985000104,None,16209,UNIVERSIDADE FEDERAL DE MINAS GERAIS,None,F,...,Não se aplica,43928.08,29959.13,2021-08-10T07:54:39,2021-08-10T07:54:39,2021-08-10T07:54:39,2021-08-10T08:00:00,2021-08-13T14:00:00,False,2021
3,15327906000072021,17217985000104-1-000003/2021,2021,3,17217985000104,None,16209,UNIVERSIDADE FEDERAL DE MINAS GERAIS,None,F,...,Não se aplica,0.00,385.00,2021-08-10T07:54:41,2021-08-10T07:54:41,2021-08-10T07:54:41,2021-08-10T08:00:00,2021-08-13T15:00:00,False,2021
4,15667806000112021,35840659000130-1-000001/2021,2021,1,35840659000130,None,81237,UNIVERSIDADE FEDERAL DE JATAI,None,F,...,Não se aplica,59188.98,NaN,2021-08-10T07:54:42,2021-08-10T07:54:42,2021-08-10T07:54:42,2021-08-10T08:00:00,2021-08-13T16:00:00,False,2021


## Dataframe com as informações gerais

In [38]:
coluna_uf = "unidadeOrgaoUfSigla"

print("Coluna de UF usada:", coluna_uf)

colunas_finais = colunas + ["anoConsulta", coluna_uf]

df = df_infos[colunas_finais].copy()

df = df.rename(columns={
    "unidadeOrgaoUfSigla": "uf"
})

print("Dimensão do DataFrame:", df.shape)

df.head()

Coluna de UF usada: unidadeOrgaoUfSigla
Dimensão do DataFrame: (6000, 14)


,idCompra,numeroCompra,anoCompraPncp,modalidadeNome,modalidadeIdPncp,unidadeOrgaoNomeUnidade,orgaoEntidadeRazaoSocial,objetoCompra,valorTotalEstimado,valorTotalHomologado,dataPublicacaoPncp,srp,anoConsulta,uf
0,78333006000012021,00001,2021,Dispensa,8,CAPITANIA DOS PORTOS DA PARAÍBA,COMANDO DA MARINHA,Contratação de Empresa especializada em Obras/...,49506.60,NaN,2021-08-10T07:54:24,False,2021,PB
1,15327906000042021,00004,2021,Dispensa,8,ESCOLA DE ENFERMAGEM/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de materiais laboratoriais e hospita...,23147.09,10684.52,2021-08-10T07:54:38,False,2021,MG
2,15329206000052021,00005,2021,Dispensa,8,INSTITUTO DE CIENCIAS EXATAS/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de material elétrico,43928.08,29959.13,2021-08-10T07:54:39,False,2021,MG
3,15327906000072021,00007,2021,Dispensa,8,ESCOLA DE ENFERMAGEM/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de açúcar cristal (pacote com 5 quil...,0.00,385.00,2021-08-10T07:54:41,False,2021,MG
4,15667806000112021,00011,2021,Dispensa,8,UNIVERSIDADE FEDERAL DE JATAI,UNIVERSIDADE FEDERAL DE JATAI,Serviço de instalação de um (1) gerador de 350...,59188.98,NaN,2021-08-10T07:54:42,False,2021,GO


## Verificação de informações faltantes e duplicados

In [39]:
print("Linhas duplicadas:", df.duplicated().sum())

print("IDs de compra duplicados:", df["idCompra"].duplicated().sum())

print("\nValores ausentes por coluna (%):")

percentual_ausentes = df.isna().mean() * 100

print(percentual_ausentes)

Linhas duplicadas: 0
IDs de compra duplicados: 224

Valores ausentes por coluna (%):
idCompra                     0.000000
numeroCompra                 0.000000
anoCompraPncp                0.000000
modalidadeNome               0.000000
modalidadeIdPncp             0.000000
unidadeOrgaoNomeUnidade      0.000000
orgaoEntidadeRazaoSocial     0.000000
objetoCompra                 0.000000
valorTotalEstimado           0.000000
valorTotalHomologado        27.166667
dataPublicacaoPncp           0.000000
srp                          0.000000
anoConsulta                  0.000000
uf                           0.000000
dtype: float64


## Limpeza dos dados

In [40]:
df_limpo = df.copy()

# Remove duplicados
antes = len(df_limpo)

df_limpo = df_limpo.drop_duplicates(subset="idCompra")

print("Duplicados removidos:", antes - len(df_limpo))

# Converte os valores para números
df_limpo["valorTotalEstimado"] = pd.to_numeric(
    df_limpo["valorTotalEstimado"],
    errors="coerce"
)

df_limpo["valorTotalHomologado"] = pd.to_numeric(
    df_limpo["valorTotalHomologado"],
    errors="coerce"
)

# Converte a data
df_limpo["dataPublicacaoPncp"] = pd.to_datetime(
    df_limpo["dataPublicacaoPncp"],
    errors="coerce"
)

# Remove registros sem UF
antes = len(df_limpo)

df_limpo = df_limpo.dropna(subset=["uf"])

print("Registros sem UF removidos:", antes - len(df_limpo))

print("\nDimensão final:", df_limpo.shape)

df_limpo.head()

Duplicados removidos: 224
Registros sem UF removidos: 0

Dimensão final: (5776, 14)


,idCompra,numeroCompra,anoCompraPncp,modalidadeNome,modalidadeIdPncp,unidadeOrgaoNomeUnidade,orgaoEntidadeRazaoSocial,objetoCompra,valorTotalEstimado,valorTotalHomologado,dataPublicacaoPncp,srp,anoConsulta,uf
0,78333006000012021,00001,2021,Dispensa,8,CAPITANIA DOS PORTOS DA PARAÍBA,COMANDO DA MARINHA,Contratação de Empresa especializada em Obras/...,49506.60,NaN,2021-08-10 07:54:24,False,2021,PB
1,15327906000042021,00004,2021,Dispensa,8,ESCOLA DE ENFERMAGEM/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de materiais laboratoriais e hospita...,23147.09,10684.52,2021-08-10 07:54:38,False,2021,MG
2,15329206000052021,00005,2021,Dispensa,8,INSTITUTO DE CIENCIAS EXATAS/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de material elétrico,43928.08,29959.13,2021-08-10 07:54:39,False,2021,MG
3,15327906000072021,00007,2021,Dispensa,8,ESCOLA DE ENFERMAGEM/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de açúcar cristal (pacote com 5 quil...,0.00,385.00,2021-08-10 07:54:41,False,2021,MG
4,15667806000112021,00011,2021,Dispensa,8,UNIVERSIDADE FEDERAL DE JATAI,UNIVERSIDADE FEDERAL DE JATAI,Serviço de instalação de um (1) gerador de 350...,59188.98,NaN,2021-08-10 07:54:42,False,2021,GO


## Estatística descritiva

In [71]:
valor = "valorTotalHomologado"

freq_uf = df_limpo["uf"].value_counts().rename("quantidade_contratacoes")

stats_valor_uf = df_limpo.groupby("uf")[valor].agg(
    total_gasto="sum",
    mediana="median",
    media="mean",
    desvio_padrao="std",
    minimo="min",
    maximo="max"
)

stats_valor_uf["coef_variacao"] = (
    stats_valor_uf["desvio_padrao"] / stats_valor_uf["media"]
)

estatisticas_uf = stats_valor_uf.copy()

estatisticas_uf["quantidade_contratacoes"] = freq_uf

estatisticas_uf = estatisticas_uf.sort_values(
    "quantidade_contratacoes",
    ascending=False
)

estatisticas_uf["total_gasto"] = estatisticas_uf["total_gasto"].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["mediana"] = estatisticas_uf["mediana"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["media"] = estatisticas_uf["media"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["desvio_padrao"] = estatisticas_uf["desvio_padrao"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["minimo"] = estatisticas_uf["minimo"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["maximo"] = estatisticas_uf["maximo"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf

,total_gasto,mediana,media,desvio_padrao,minimo,maximo,coef_variacao,quantidade_contratacoes
uf,,,,,,,,
SP,"R$ 635.819.084,37","8.400,00","648.133,62","7.593.433,02","0,00","137.035.308,92",11.715845,1175
RJ,"R$ 173.004.835,14","6.960,00","294.727,15","2.691.632,75","0,00","35.091.700,00",9.132626,894
DF,"R$ 1.074.261.182,27","10.500,00","2.992.370,98","48.538.644,21","0,02","918.700.100,00",16.220798,468
RS,"R$ 48.356.946,27","3.544,00","155.990,15","1.430.460,70","35,40","21.548.888,74",9.170199,434
MG,"R$ 187.248.944,63","8.621,10","590.690,68","7.042.798,70","0,00","123.176.842,20",11.922989,416
PR,"R$ 8.644.750,72","6.351,00","41.165,48","147.268,70","109,99","1.676.830,14",3.577480,287
BA,"R$ 15.454.113,27","12.158,27","107.320,23","600.743,74","0,00","6.256.165,55",5.597675,208
GO,"R$ 9.067.757,53","18.143,70","77.502,20","286.416,58","2,00","2.148.000,00",3.695593,191
RO,"R$ 9.497.458,33","19.080,00","123.343,61","362.678,52","0,01","2.022.000,00",2.940392,188
